In [ ]:
!pip install torchgeometry

In [2]:
#!/usr/bin/env python
# coding: utf-8

"""
Example: Running FrankMocap inference on a sample video in a Jupyter notebook.
This version includes integrated error handling for CUDA out-of-memory errors.
"""

import sys
import os
import cv2
import numpy as np
import torch
import plotly.graph_objects as go
from IPython.display import HTML, display

# 1. Add FrankMocap to the Python path
frankmocap_repo_path = "/home/robotics/Desktop/frankmocap"
sys.path.append(frankmocap_repo_path)

# Set the current working directory to the repository root.
os.chdir(frankmocap_repo_path)

# 2. Import FrankMocap modules
try:
    from bodymocap.body_mocap_api import BodyMocap  # Example import
    # from mocap_utils.demo_utils import save_output  # Pseudo function (commented out)
except ImportError as e:
    print("Could not import FrankMocap modules. Check paths or environment.")
    raise e

# 3. Paths: input video, output dir
video_input_path = os.path.join(frankmocap_repo_path, "sample_data", "sample_video.mp4")  # your input video
output_dir = os.path.join(frankmocap_repo_path, "sample_data", "output_frankmocap")
os.makedirs(output_dir, exist_ok=True)

# 4. Initialize FrankMocap body model
# Set device to "cuda" if available, otherwise "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

# If using GPU, clear CUDA cache
if device == "cuda":
    torch.cuda.empty_cache()

# Define the SMPL directory path (from your repository tree)
smpl_dir = os.path.join(frankmocap_repo_path, "extra_data", "smpl")

# Attempt to initialize BodyMocap on the current device.
# If a CUDA out-of-memory error occurs, switch to CPU.
try:
    body_mocap = BodyMocap(
        regressor_checkpoint=os.path.join(frankmocap_repo_path, "extra_data", "body_module", "pretrained_weights", "YOUR_BODY_MODEL.pth"),
        device=device,
        smpl_dir=smpl_dir
    )
except RuntimeError as e:
    if "CUDA error: out of memory" in str(e):
        print("CUDA out of memory error during model initialization. Switching to CPU.")
        device = "cpu"
        body_mocap = BodyMocap(
            regressor_checkpoint=os.path.join(frankmocap_repo_path, "extra_data", "body_module", "pretrained_weights", "YOUR_BODY_MODEL.pth"),
            device=device,
            smpl_dir=smpl_dir
        )
    else:
        raise e

print("Loaded FrankMocap body model (example).")

# 5. Process the video frame-by-frame
cap = cv2.VideoCapture(video_input_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_video_path = os.path.join(output_dir, "frankmocap_output.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

all_3d_smpl_joints = []  # for storing 3D results if needed
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 6. Perform FrankMocap body inference (using a placeholder function)
    try:
        # 'body_mocap.process_frame' is a pseudo name; check FrankMocap docs
        smpl_output = body_mocap.process_frame(frame_rgb)
        # Expected: smpl_output contains a key 'pred_joints_smpl' with shape (Nx24x3)
        pred_joints_3d = smpl_output['pred_joints_smpl']
    except Exception as e:
        print(f"Error during processing frame {frame_idx}: {e}")
        pred_joints_3d = np.zeros((1, 24, 3))

    # Save 3D info (for the first person detected)
    if len(pred_joints_3d) > 0:
        all_3d_smpl_joints.append(pred_joints_3d[0])

    # 7. Visualize skeleton in 2D (placeholder code)
    overlayed_frame = frame  # Replace with actual visualization if available

    out_writer.write(overlayed_frame)
    frame_idx += 1

cap.release()
out_writer.release()

print(f"Processed {frame_idx} frames. Output video saved to {output_video_path}")

# 8. Plotly 3D Visualization of the final frame's SMPL joints
if len(all_3d_smpl_joints) > 0:
    final_frame_joints = all_3d_smpl_joints[-1]  # shape (24, 3)
    x_vals = final_frame_joints[:, 0]
    y_vals = final_frame_joints[:, 1]
    z_vals = final_frame_joints[:, 2]

    fig = go.Figure(data=[go.Scatter3d(
        x=x_vals,
        y=y_vals,
        z=z_vals,
        mode='markers+lines',
        marker=dict(size=4, color='blue'),
        name='FrankMocap_Body_3D'
    )])
    fig.update_layout(
        title="FrankMocap 3D Body Pose (Example)",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        )
    )
    fig.show()

    # Optionally, save as HTML
    plot_save_path = os.path.join(output_dir, "frankmocap_3d_body.html")
    fig.write_html(plot_save_path)
    print(f"3D plot saved to {plot_save_path}")
else:
    print("No 3D joints found to visualize.")


Running on device: cuda
CUDA out of memory error during model initialization. Switching to CPU.


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
